# Prática: Regressão Logística e Gradient Descent
# Insper AI

---

O objetivo dessa atividade é implementar e treinar manualmente um classificador binário usando Regressão Logística e Gradiente Descendente.

### Preparação inicial: importando bibliotecas, dados e pré-processamento

Rode:

```bash
uv sync
```

Não precisa fazer nada nessa seção. Apenas execute as células.

In [24]:
# Importar as bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer(as_frame=True)
df = cancer.frame

In [25]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Tamanho do conjunto de treino:", X_train.shape, y_train.shape)
print("Quantidade de features:", X_train.shape[1])
print("Tamanho do conjunto de teste:", X_test.shape, y_test.shape)

Tamanho do conjunto de treino: (455, 30) (455,)
Quantidade de features: 30
Tamanho do conjunto de teste: (114, 30) (114,)


---
### A Teoria da Regressão Logística

Diferente da Regressão Linear, que prevê um **valor contínuo**, a Regressão Logística prevê uma **probabilidade**, que é então usada para classificar a entrada em uma de duas categorias. Portanto, a variável target é binária (ex: Sim/Não, Doente/Saudável, 0/1).

#### O Desafio: De Linha a Probabilidade

Se usássemos a equação da Regressão Linear ($\hat{y} = \mathbf{w}^T \cdot \mathbf{x} + b$) para um problema de classificação, teríamos um problema: a saída pode ser qualquer número real (de $-\infty$ a $+\infty$), e não uma probabilidade que deve, por definição, estar entre 0 e 1.

A solução da Regressão Logística é aplicar uma função "esmagadora" ao resultado da equação linear. Essa função é chamada de **Função Sigmoide** (ou Função Logística), que é um exemplo de **função de ativação**.

#### Função de Ativação
Uma função de ativação é usada em modelos de machine learning para **introduzir não-linearidade**, permitindo que o modelo capture relações complexas nos dados. No caso da Regressão Logística, a função de ativação transforma a saída linear em uma probabilidade, facilitando a classificação.

#### Função Sigmoide

A Função de Ativação Sigmoide, denotada por $\sigma(z)$, tem a seguinte fórmula:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Ela pega qualquer valor de entrada $z$ e o transforma em um valor entre 0 e 1, que podemos interpretar como uma probabilidade.

![Função Sigmoide](https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Logistic-curve.svg/600px-Logistic-curve.svg.png)

#### A Equação Completa

A equação da Regressão Logística combina a equação linear com a função sigmoide:

1.  Primeiro, calcula-se uma pontuação linear (exatamente como na Regressão Linear):
    $$z = \mathbf{w}^T \cdot \mathbf{x} + b$$

2.  Em seguida, essa pontuação é passada pela função sigmoide para obter a probabilidade:
    $$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-(\mathbf{w}^T \cdot \mathbf{x} + b)}}$$

Onde:
* $\hat{p}$: a probabilidade prevista de a amostra pertencer à classe positiva (classe 1).
* $\mathbf{w}$ e $b$: são os vetores de pesos e bias que o modelo precisa aprender.

O "aprendizado" aqui significa encontrar os melhores valores de $\mathbf{w}$ e $b$ que fazem o modelo prever probabilidades altas para as amostras da classe positiva e probabilidades baixas para as da classe negativa.

A previsão final do modelo é feita com base em um limiar (threshold), geralmente 0.5:
- Se $\hat{p} \geq 0.5$, classifica como classe 1 (classe positiva).
- Se $\hat{p} < 0.5$, classifica como classe 0 (classe negativa).

### Implementação

#### Passo 1: Inicialização dos Parâmetros
A primeira etapa é inicializar os pesos $\mathbf{w}$ e o bias $b$. Uma prática comum é iniciar os pesos com valores pequenos aleatórios e o bias com zero.

**Dicas**:
- Crie uma função `initialize_parameters(n_features)` que retorna um vetor de pesos e um bias inicializados.

- Use `numpy` para criar arrays de pesos.
- Inicialize os pesos com valores aleatórios pequenos (ex: `np.random.randn(n_features) * 0.01`).

In [26]:
# SEU CÓDIGO AQUI

def initialize_parameters(n_features):
    """
    Inicializa os pesos e o bias.
    
    Argumentos:
    n_features -- número de features no conjunto de dados
    
    Retorna:
    w -- vetor de pesos inicializados
    b -- bias inicializado
    """
    np.random.seed(42)  # Para reprodutibilidade
    w = np.random.randn(n_features) * 0.01  # Pesos pequenos aleatórios
    b = 0.0  # Bias inicializado como zero
    return w, b

#### Passo 2: Função Sigmoide
Implemente a função sigmoide que transforma a saída linear em uma probabilidade.

**Dicas**:
- Crie uma função `sigmoid(z)` que retorna a sigmoide de `z`.

- Use `numpy` para calcular a sigmoide de forma vetorizada.
- Use a função `np.exp()` para calcular a exponencial.

- Reflita sobre o uso da função `np.clip()` no código fornecido. Por que é importante limitar os valores de `z`?

In [27]:
def sigmoid(z):
    """
    Calcula a função sigmoide.
    
    Argumentos:
    z -- valor escalar ou array numpy
    
    Retorna:
    s -- resultado da função sigmoide aplicada a z
    """
    z = np.clip(z, -500, 500)
    s = 1 / (1 + np.exp(-z))
    return s

#### Passo 3: Fazendo previsões
Implemente a função que faz as previsões usando os pesos e bias atuais.


**Dicas**:
- Crie uma função `predict(X, w, b)` que retorna uma **lista** com as previsões das probabilidades para um conjunto de dados `X`.  

- Use a função `sigmoid()` para calcular as probabilidades.  
- Use a função np.dot() para calcular o produto escalar entre `X` e `w` (lembre-se que `w` é um vetor de pesos, um para cada feature de `X`). Isso possibilita a implementação da função sem loops (substancialmente mais eficiente e é a prática padrão). 

    Se você não conhece ou lembra, pesquise o que é e como funciona um **produto escalar**. 

In [28]:
def predict(X, w, b):
    """
    Faz previsões usando os pesos e bias atuais.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    y_pred -- previsões (numpy array de shape (m,))
    """
    z = np.dot(X, w) + b  # cálculo do produto escalar + bias
    probabilities = sigmoid(z)  # cálculo das probabilidades
    return probabilities

---

### A Função de Custo

Não podemos usar o Erro Quadrático Médio (MSE) da Regressão Linear, pois quando combinado com a função sigmoide, ele cria uma função de custo não-convexa, cheia de mínimos locais, dificultando a otimização.

Para a Regressão Logística, usamos uma função de custo chamada **Log Loss**, ou **Entropia Cruzada Binária (Binary Cross-Entropy)**. A intuição por trás dela é simples: ela penaliza fortemente o modelo quando ele está "confiantemente errado".

* Se a classe real é 1, a função de custo penaliza o modelo à medida que a probabilidade prevista $\hat{p}$ se aproxima de 0.
* Se a classe real é 0, a função de custo penaliza o modelo à medida que a probabilidade prevista $\hat{p}$ se aproxima de 1.

A fórmula completa que combina os dois casos é:

$$J(\mathbf{w}, b) = - \frac{1}{m} \sum_{i=1}^{m} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

Onde:
- $m$: número de exemplos no conjunto de dados.
- $y_i$: classe real do exemplo $i$ (0 ou 1).
- $\hat{y}_i$: probabilidade prevista pelo modelo para o exemplo $i$.

Assim como antes, a função de custo depende dos parâmetros do modelo ($\mathbf{w}$ e $b$). Treinar o modelo significa encontrar os valores ótimos de $\mathbf{w}$ e $b$ que minimizam essa função de custo, e isso também é feito através de algoritmos de otimização como o **Gradiente Descendente**.

Os vídeos 34 e 35 do curso de Machine Learning devem ajudar a entender a função de custo Log Loss: https://www.youtube.com/watch?v=vq4Ie5xWhww&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=34

### Implementação
Implemente uma função que calcula a função de custo Log Loss.

**Dicas**:
- Crie uma função `cost_function(X, y, w, b)` que retorna o valor da função de custo.

- Use a função `predict()` para obter as probabilidades previstas.
- Use a função `np.log()` para calcular o logaritmo.
- Você pode implementar a somatória usando um loop ou de forma vetorizada com `np.sum()`.

- IMPORTANTE: Para evitar problemas de log(0), você pode adicionar um pequeno valor (ex: `1e-15`) às probabilidades previstas antes de calcular o logaritmo.

In [29]:
def cost_function(X, y, w, b):
    """
    Calcula a função de custo logístico.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    y -- rótulos verdadeiros (numpy array de shape (m,))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    cost -- valor da função de custo
    """
    m = X.shape[0]  # número de exemplos
    probabilities = predict(X, w, b)  # previsões do modelo
    # Cálculo da função de custo logístico
    cost = - (1/m) * np.sum(y * np.log(probabilities + 1e-15) + (1 - y) * np.log(1 - probabilities + 1e-15))
    return cost

---
### Otimização: Gradiente Descendente

#### Introdução
O Gradiente Descendente é um algoritmo de otimização usado para **minimizar a função de custo** ajustando os parâmetros do modelo ($\mathbf{w}$ e $b$). 

A ideia básica é calcular o **gradiente da função de custo** em relação aos parâmetros e, em seguida, atualizar os parâmetros na direção oposta ao gradiente.

#### O gradiente da função de custo
O gradiente é um vetor que aponta na **direção de maior crescimento** de uma função, sendo composto pelas derivadas parciais em relação a cada parâmetro.

**Derivada parcial**: Assim como a derivada representa a **taxa de crescimento** de uma função, a derivada parcial representa a taxa em relação a um parâmetro específico. Ou seja, se fixarmos todos os parâmetros, exceto um, a derivada parcial nos dirá como a função de custo muda quando alteramos apenas aquele parâmetro.

   $$\nabla J(\mathbf{w}, b) = \left[ \frac{\partial J}{\partial \mathbf{w}}, \frac{\partial J}{\partial b} \right]$$

Onde:
- $\nabla$ é o símbolo do gradiente.
- $\frac{\partial }{\partial \mathbf{w}}$ representa a derivada parcial em relação ao parâmetro correspondente (nesse caso $\mathbf{w}$).

Para a Regressão Logística, as derivadas parciais da função de custo Log Loss são:

$$\frac{\partial J}{\partial \mathbf{w_j}} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i) \cdot \mathbf{x_j}_i$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)$$

Onde:
- $m$: número de exemplos no conjunto de dados.
- $i$: índice do exemplo atual.
- $j$: índice da feature atual.

Fique à vontade para deduzir essas fórmulas, mas não é obrigatório para a implementação.

O vídeo 36 do curso de Machine Learning deve ajudar a entender as derivadas parciais: https://www.youtube.com/watch?v=H9bXvYh3nO4&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=36

#### Método do Gradiente Descendente

Como você pode ter imaginado, o método é iterativo, e consiste nos seguintes passos:

1. **Inicialização**: Começamos com valores iniciais para $\mathbf{w}$ e $b$ (geralmente zeros ou pequenos valores aleatórios).

2. **Cálculo do Gradiente**: Para cada iteração, calculamos o gradiente da função de custo:

   $$\nabla J(\mathbf{w}, b) = \left[ \frac{\partial J}{\partial \mathbf{w}}, \frac{\partial J}{\partial b} \right]$$

3. **Atualização dos Parâmetros**: Atualizamos os parâmetros na direção oposta ao gradiente:

   $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}}$$
   $$b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

   Onde $\alpha$ é a taxa de aprendizado, um hiperparâmetro que controla o tamanho dos passos de atualização.

4. **Iteração**: Repetimos os passos 2 e 3 até que a função de custo converja (ou seja, até que as atualizações se tornem muito pequenas) ou até atingir um número máximo de iterações.

Os vídeos 15 a 17 do curso de Machine Learning devem ajudar nessa parte: https://www.youtube.com/watch?v=WtlvKq_zxPI&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=16

### Implementação

#### Passo 1: Cálculo do Gradiente
Implemente uma função que calcula o gradiente da função de custo em relação aos parâmetros $\mathbf{w}$ e $b$.

**Dicas**:
- Crie uma função `gradient(X, y, w, b)` que retorna as derivadas parciais $\frac{\partial J}{\partial \mathbf{w}}$ e $\frac{\partial J}{\partial b}$.

- Use a função `predict()` para obter as probabilidades previstas. Lembre-se que predict retorna um **array** (lista) de probabilidades.
- Lembre-se que cada elemento de `w`, ou seja, cada peso de uma feature, tem sua própria derivada parcial.
- Crie uma lista ou array para armazenar as derivadas parciais de cada peso.

- Implemente a função de forma vetorizada usando `np.dot()` para eficiência, assim como você fez na função `predict()`.  Se você não fez, volte e implemente. Se você não conhece, procure o que é e como funciona um **produto escalar**.

In [30]:
def gradient_loop(X, y, w, b):
    m = X.shape[0]  # número de amostras
    n = X.shape[1]  # número de features
    dj_dw = np.zeros(n)  # inicializa o vetor de derivadas parciais para w
    dj_db = 0.0  # inicializa a derivada parcial para b

    # Calcula as previsões
    y_hat = predict(X, w, b) 

    # Calcula as derivadas parciais
    for i in range(m):
        error = y_hat[i] - y[i]
        for j in range(n):
            dj_dw[j] += error * X[i, j]
        dj_db += error

    # Média das derivadas parciais
    dj_dw /= m
    dj_db /= m

    return dj_dw, dj_db

In [31]:
def gradient(X, y, w, b):
    """
    Calcula o gradiente da função de custo de forma vetorizada.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    y -- rótulos verdadeiros (numpy array de shape (m,))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    dj_dw -- vetor de derivadas parciais em relação a w
    dj_db -- derivada parcial em relação a b
    """
    m = X.shape[0]  # número de exemplos
    y_hat = predict(X, w, b)  # previsões do modelo
    
    error = y_hat - y  # erro entre previsões e rótulos verdadeiros
    
    dj_dw = (1/m) * np.dot(X.T, error)  # derivada parcial em relação a w
    dj_db = (1/m) * np.sum(error)  # derivada parcial em relação a b
    
    return dj_dw, dj_db

#### Passo 2: Implementação do Gradiente Descendente
Implemente a função que executa o algoritmo do Gradiente Descendente para otimizar os parâmetros $\mathbf{w}$ e $b$.

**Dicas**:
- Crie uma função `gradient_descent(X, y, w_inicial, b_inicial, alpha, num_iterations)` que retorna os parâmetros otimizados.
- Use a função `gradient()` para calcular as derivadas parciais.
- Atualize os parâmetros $\mathbf{w}$ e $b$ usando as fórmulas de atualização.
- Use a função `cost_function()` para calcular o valor da função de custo a cada iteração e armazene esses valores para análise posterior.
- Adicione prints para mostrar o valor da função de custo a cada 100 iterações (ou outro intervalo que você achar interessante).
- Considere adicionar um critério de parada baseado na convergência da função de custo (ex: se a mudança na função de custo for menor que um certo limiar).

In [32]:
def gradient_descent(X, y, w_inicial, b_inicial, alpha, num_iterations):
    J_history = []  # para armazenar o histórico da função de custo
    w = w_inicial.copy()
    b = b_inicial

    for i in range(num_iterations):
        # Calcula o gradiente
        dj_dw, dj_db = gradient(X, y, w, b)

        # Atualiza os parâmetros
        w -= alpha * dj_dw
        b -= alpha * dj_db

        # Calcula e armazena o custo
        cost = cost_function(X, y, w, b)
        J_history.append(cost)

        # Opcional: imprime o custo a cada 100 iterações
        if i % 100 == 0:
            print(f"Iteration {i}: Cost {cost}")

        if i > 0 and abs(J_history[-2] - J_history[-1]) < 1e-6:
            print(f"Converged after {i} iterations.")
            break

    return w, b, J_history

#### Passo 3: Aplicando o Gradiente Descendente
Agora, com todas as funções implementadas, treine o modelo usando o Gradiente Descendente.

**Dicas**:
- Use a função `initialize_parameters()` para obter os pesos e bias iniciais.
- Defina uma taxa de aprendizado (ex: `alpha = 0.001`) e um número máximo de iterações (ex: `num_iterations = 10000`, o mesmo do outro notebook).
- Chame a função `gradient_descent()` com os parâmetros apropriados para treinar o modelo.
- Após o treinamento, imprima os pesos e bias finais.

In [33]:
w, b = initialize_parameters(X_train.shape[1])
alpha = 0.001
num_iterations = 10000

w, b, J_history = gradient_descent(X_train.values, y_train.values, w, b, alpha, num_iterations)

print("\nPesos finais:", w)
print("Bias final:", b)

Iteration 0: Cost 12.82868837525254
Iteration 100: Cost 9.923713796084742
Iteration 200: Cost 2.928244298077806
Iteration 300: Cost 3.2028231308094854
Iteration 400: Cost 3.8595020125421517
Iteration 500: Cost 2.578464464678371
Iteration 600: Cost 2.5494498921351756
Iteration 700: Cost 2.5251664954772615
Iteration 800: Cost 2.563093329001853
Iteration 900: Cost 2.5507321063673905
Iteration 1000: Cost 2.4417041393627956
Iteration 1100: Cost 2.3693188660896194
Iteration 1200: Cost 2.341990905028086
Iteration 1300: Cost 2.3139875915426504
Iteration 1400: Cost 8.347542002472546
Iteration 1500: Cost 2.3630550548565736
Converged after 1542 iterations.

Pesos finais: [ 3.61447631e-01  3.44213668e-01  2.01755943e+00  7.01534925e-01
  4.69672514e-04 -5.73053051e-03  6.81134202e-03  3.98354042e-03
  6.32583048e-04  7.81662024e-03 -3.88659727e-03  2.15537495e-02
 -1.82070622e-02 -9.84181726e-01 -1.71503107e-02 -6.53055639e-03
 -1.15734322e-02  2.88619426e-03 -8.71806758e-03 -1.41189517e-02
  3.92

#### Compare

Você deve ter exibido os parâmetros do modelo treinado pelo scikit-learn no outro notebook. Compare esses parâmetros com os que você obteve manualmente. Eles devem ser muito semelhantes, embora possam não ser exatamente iguais devido a diferenças na inicialização, taxa de aprendizado, número de iterações e outros fatores.

--- 
### Avaliação do Modelo

Após treinar o modelo, é importante avaliar seu desempenho usando métricas apropriadas para problemas de classificação binária.

**Dicas**:
- Use a função `predict()` para obter as previsões do modelo no conjunto de teste.
- Calcule a acurácia, matriz de confusão, precisão, recall e F1-score. Sinta-se à vontade para usar as funções do `sklearn.metrics` para facilitar esses cálculos.
- Compare as previsões com as classes reais para calcular as métricas.

In [34]:
# SEU CÓDIGO AQUI
y_pred = predict(X_test, w, b)
y_pred_classes = (y_pred >= 0.5).astype(int)  # Convertendo probabilidades em classes (0 ou 1)

# MÉTRICAS DE AVALIAÇÃO
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calculando as métricas
accuracy = accuracy_score(y_test, y_pred_classes)
precision = precision_score(y_test, y_pred_classes)
recall = recall_score(y_test, y_pred_classes)
f1 = f1_score(y_test, y_pred_classes)

# Exibindo os resultados
print(f"Acurácia: {accuracy:.4f}")
print(f"Precisão: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Acurácia: 0.9474
Precisão: 0.9577
Recall: 0.9577
F1-Score: 0.9577


### Conclusão

Neste notebook, você implementou um classificador de Regressão Logística a partir do zero, utilizando o algoritmo de otimização de Gradiente Descendente (Batch) para treinar os parâmetros do modelo. Compare o desempenho e os parâmetros do modelo manual com a implementação padrão da biblioteca `scikit-learn` feita no outro notebook.

**Principais Aprendizados:**

* **Abertura da "Caixa-Preta":** A atividade demonstrou na prática o mecanismo interno de um classificador linear e esclareceu o processo de "treinamento":
    1.  Definição de uma **Função de Custo** (Log Loss) para quantificar o erro.
    2.  Cálculo do **Gradiente** para determinar a direção de maior erro.
    3.  Ajuste iterativo dos **Parâmetros** ($\mathbf{w}$ e $b$) na direção oposta ao gradiente para minimizar o custo.

* **Teoria vs. Código:** A relação entre a teoria matemática (cálculo das derivadas parciais) e a implementação prática (código em Python/NumPy) foi estabelecida de forma concreta.

* **Impacto dos Hiperparâmetros:** A importância de hiperparâmetros como a **taxa de aprendizado ($\alpha$)** e o **número de iterações** se tornou tangível, impactando diretamente a convergência e o desempenho final do modelo.

Este conhecimento é fundamental para entender o funcionamento interno de um neurônio artificial e seu processo de otimização (Gradiente Descendente), que é a base fundamental para o próximo encontro, onde construiremos a nossa primeira Rede Neural.